In [ ]:
"""
导包
""" 


import os
from pathlib import Path
from sdp.dataset.base_dataset import BaseDataset
from sdp.reader.reader_factory import ReaderFactory
from sdp.dataset.dataset_factory import DatasetFactory
from sdp.processor.processor_factory import ProcessorFactory
from sdp.processor.processors.HwProcessor import HwProcessor
from sdp.processor.processors.BfeeProcessor import BfeeProcessor

In [ ]:
from sdp.processor.processors.WiproxProcessor import WiproxProcessor

"""
主函数
"""
# 主函数
def process_file_from_folder(**kwargs) -> BaseDataset:
    folder_path = kwargs.get('folder_path', '')
    task_type = kwargs.get('task_type', '')
    final_fs = kwargs.get('final_fs', 1000)
    num_samples = kwargs.get('num_samples', 100000)
    global res
    
    folder_path = Path(folder_path)
    if not folder_path.exists() or not folder_path.is_dir():
        raise ValueError(f"无效的文件夹路径: {folder_path}")
    files = [f for f in folder_path.rglob("*") if f.is_file() and "truth" not in f.name]
    if not files:
        print(f"文件夹 {folder_path} 中没有文件")
        return None
    
    # 使用第一个文件确定主格式
    sample_file = files[0]
    reader = ReaderFactory.create_reader(str(sample_file))
    sample_frame = reader.read_file(sample_file).frames[0]
    processor = ProcessorFactory.get_processor(sample_frame)
    

    print(f"检测到主文件格式: {type(reader).__name__}")
    
    print(f"开始处理 {len(files)} 个文件...")
    
    csi_data_list = []
    # 处理所有文件
    for file_path in files:
        try:
            csi_data = reader.read_file(str(file_path))
            csi_data_list.append(csi_data)
            
            print(f"√ 已处理: {file_path.name}")
        
        except Exception as e:
            print(f"× 处理失败 {file_path.name}: {str(e)}")
    
    print(f"处理完成! 共处理 {len(files)} 个文件")
    
    # 情况处理'
    if type(processor) == HwProcessor:
        res = list(processor.process(csi_data_list, folder_path=folder_path))
    elif type(processor) == BfeeProcessor:
        res = processor.process(csi_data_list, folder_path=folder_path, task_type=task_type, final_fs=final_fs)
    elif type(processor) == WiproxProcessor:
        res = processor.process(csi_data_list, folder_path=folder_path, num_samples=num_samples)
    pprint(res)
    # 构造dataset
    dataset = DatasetFactory.create_dataset(res, reader)
    
    return dataset

In [ ]:
from pprint import pprint

"""
执行位置
"""
PROJECT_ROOT = Path.cwd()
os.chdir(PROJECT_ROOT)

# data path
folder_path = PROJECT_ROOT / "data/hw_data"

# param for BfeeProcessor
task_type = 'Gesture Recognition'
final_fs = 1000

# param for Wi_prox_Processor
num_samples = 100000

dataset = process_file_from_folder(folder_path=folder_path, task_type=task_type, final_fs=final_fs, num_samples=num_samples)
# pprint(f"the final dataset: {vars(dataset)}")
print("process success")